In [1]:
pip install fredapi python-dotenv pandas



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [71]:
from fredapi import Fred
from dotenv import load_dotenv
import pandas as pd
import os
from functools import reduce

In [7]:
load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
fred_auth = fred = Fred(api_key=FRED_API_KEY)

# Point In Time Logic (Predicting on a monthly time unit)

* If your model is **designed to run on the 1st of the month** (e.g., Jan 1st), it has to use whatever the most recently published data is on that exact day.
* When we train our model using historical data, we are not just trying to find historical patterns. We are trying to simulate exactly what the model would have seen if it had been running live on that historical day.

### Example
* When we are training the model, its supposed to predict on a time period of one month (let's say the 31st of each month)
* When it predicts on the 31st of the month you are try to fetch all the data, but for january's unemployment rate, cpi, etc. all of these things aren't available until mid feb
* So actually, you are supposed to use last month's data to do the prediction (e.g. the december data is used for january prediction)

In [94]:
recession_data = pd.DataFrame(fred_auth.get_series("USREC")).reset_index().rename(columns={"index": "aligned_date", 0: "recession"})
recession_data = recession_data[recession_data['aligned_date'] >= pd.to_datetime('2001-01-01')]

# list of metrics to pull from FRED: consumer price index, payroll employment, unemployment rate, industrial production, 10 year minus 3 month treasury yield
# 'realtime_start' = The Publication Day / Knowledge Date
# 'date' = The Observation Month
# 'value' = The CPI value at that time

monthly_revised = ["CPIAUCSL", "PAYEMS", "UNRATE", "INDPRO"]
daily_market = ["T10Y3M"]

economic_data = {}

economic_data["recession"] = recession_data

for metric in monthly_revised:
    print(f"Pulling first-release data for {metric}...")
    df = fred_auth.get_series_all_releases(metric, realtime_start="2001-01-01")
    df = df[df['date'] >= pd.to_datetime('2001-01-01')]
    # The reason for first releases is because we are assessing the data as it was available at the time of release
    # because this often guides future economic decisions and market reactions.
    # we also only keep the 'realtime_start' and 'value' columns, as we our focusing on the publication day
    first_release_df = df.groupby("date").head(1)
    first_release_df["aligned_date"] = pd.to_datetime(first_release_df["realtime_start"]) + pd.offsets.MonthBegin(1)
    first_release_df.rename(columns={"realtime_start": "publication_date",'date': 'observation_month','value': f"{metric}"}, inplace=True)
    economic_data[metric] = first_release_df


for metric in daily_market:
    print(f"Pulling market data for {metric}...")
    economic_data[metric] = fred_auth.get_series(metric, observation_start="2001-01-01")
    # resample for last day of the month
    # convert to period and back to timestamp to get the last day of the month, 
    # reset index and rename columns
    economic_data[metric] = (
        pd.DataFrame(economic_data[metric])
        .resample("ME")
        .last()
        .reset_index()
        .rename(columns={"index": "date", 0: f"{metric}"})
    )
    economic_data[metric]["aligned_date"] = economic_data[metric]["date"] + pd.offsets.MonthBegin(1)

Pulling first-release data for CPIAUCSL...
Pulling first-release data for PAYEMS...
Pulling first-release data for UNRATE...
Pulling first-release data for INDPRO...
Pulling market data for T10Y3M...


In [95]:
economic_data.items()

dict_items([('recession',      aligned_date  recession
1753   2001-01-01        0.0
1754   2001-02-01        0.0
1755   2001-03-01        0.0
1756   2001-04-01        1.0
1757   2001-05-01        1.0
...           ...        ...
2052   2025-12-01        0.0
2053   2026-01-01        0.0
2054   2026-02-01        0.0
2055   2026-03-01        0.0
2056   2026-04-01        0.0

[304 rows x 2 columns]), ('CPIAUCSL',          publication_date    observation_month CPIAUCSL aligned_date
856   2001-02-21 00:00:00  2001-01-01 00:00:00    175.7   2001-03-01
858   2001-03-21 00:00:00  2001-02-01 00:00:00    176.2   2001-04-01
862   2001-04-17 00:00:00  2001-03-01 00:00:00    176.3   2001-05-01
866   2001-05-16 00:00:00  2001-04-01 00:00:00    176.8   2001-06-01
871   2001-06-15 00:00:00  2001-05-01 00:00:00    177.5   2001-07-01
...                   ...                  ...      ...          ...
2387  2025-12-18 00:00:00  2025-11-01 00:00:00  325.031   2026-01-01
2389  2026-01-13 00:00:00  2025-12-

- The model now assumes we are running on the [first of each month]
- We pull the most 'recent' available data for cpi/unemployment date, which is the data published on the month before - which was observing for the month before that
- We then also pull the last known yield curve data excluding the day we are running it
- Which is why we have aligned dates for all the data sources which is offset to the first day of the month after the data was published

In [96]:
all_dfs_cleaned = [df[['aligned_date', metric]] for metric, df in economic_data.items()]
df_final = reduce(lambda left, right: pd.merge(left, right, on='aligned_date', how='outer'), all_dfs_cleaned)

In [97]:
df_final.head(20)

,aligned_date,recession,CPIAUCSL,PAYEMS,UNRATE,INDPRO,T10Y3M
0,2001-01-01,0.0,NaN,NaN,NaN,NaN,NaN
1,2001-02-01,0.0,NaN,NaN,NaN,NaN,0.20
2,2001-03-01,0.0,175.7,132129.0,4.2,146.997,0.07
3,2001-04-01,1.0,176.2,132237.0,4.2,146.001,0.63
4,2001-05-01,1.0,176.3,132221.0,4.3,146.466,1.40
5,2001-06-01,1.0,176.8,132027.0,4.5,144.866,1.80
6,2001-07-01,1.0,177.5,132453.0,4.4,143.12,1.77
7,2001-08-01,1.0,177.9,132383.0,4.5,142.501,1.53
8,2001-09-01,1.0,177.4,132395.0,4.5,142.849,1.48
9,2001-10-01,1.0,177.5,132331.0,4.9,141.499,2.20
